# LoRA and QLoRA from Scratch

Low-Rank Adaptation (LoRA) and its quantized variant (QLoRA): the dominant parameter-efficient fine-tuning methods for LLMs.

**Interview questions:**
- "Why does low-rank adaptation work?"
- "How many parameters does LoRA add?"
- "How do you choose the rank?"
- "Which layers should LoRA be applied to?"

---

## Key Idea

Instead of updating a full weight matrix $W \in \mathbb{R}^{d \times d}$ during fine-tuning, LoRA learns a low-rank update:

$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, and $r \ll d$.

| Method | Trainable Params (d=4096, r=16) | Memory |
|--------|-------------------------------|--------|
| Full fine-tuning | $d^2 = 16M$ | Full optimizer states |
| **LoRA** | $2 \cdot d \cdot r = 131K$ | **~0.8% of full** |
| **QLoRA** | Same as LoRA | **4-bit base + LoRA** |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from copy import deepcopy

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. LoRA Layer Implementation

The core idea: freeze $W$, learn low-rank matrices $A$ and $B$.

- $A$ is initialized from $\mathcal{N}(0, \sigma^2)$ (Kaiming)
- $B$ is initialized to zero (so $\Delta W = 0$ at start)
- Scaling factor $\alpha / r$ controls the magnitude of the update

In [ ]:
class LoRALayer(nn.Module):
    """Low-Rank Adaptation layer.
    
    Wraps an existing nn.Linear and adds a low-rank update: h = Wx + (alpha/r) * BAx
    Only A and B are trainable; W is frozen.
    """
    def __init__(self, base_layer: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        d_out, d_in = base_layer.weight.shape
        
        # Freeze base weights
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False
        
        # LoRA matrices: W' = W + (alpha/r) * B @ A
        # A: d_in -> rank (down-projection)
        # B: rank -> d_out (up-projection)
        self.lora_A = nn.Parameter(torch.empty(rank, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        
        # Initialize A with Kaiming uniform
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base forward (frozen)
        base_out = self.base_layer(x)
        # LoRA forward: x -> A -> B -> scale
        lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B)
        return base_out + self.scaling * lora_out
    
    def merge_weights(self) -> nn.Linear:
        """Merge LoRA weights back into base layer for inference.
        Returns a standard nn.Linear with no overhead."""
        merged = deepcopy(self.base_layer)
        merged.weight.data += self.scaling * (self.lora_B @ self.lora_A)
        merged.weight.requires_grad = False
        return merged
    
    @property
    def num_trainable_params(self):
        return self.lora_A.numel() + self.lora_B.numel()

# Quick test
base = nn.Linear(512, 512)
lora = LoRALayer(base, rank=8, alpha=16)
x = torch.randn(2, 10, 512)

print(f"Base params: {base.weight.numel() + base.bias.numel():,}")
print(f"LoRA params: {lora.num_trainable_params:,}")
print(f"Ratio: {lora.num_trainable_params / (base.weight.numel() + base.bias.numel()) * 100:.2f}%")
print(f"Output shape: {lora(x).shape}")

## 2. Inject LoRA into Transformer Attention

Standard practice: apply LoRA to the query and value projections ($W_Q$, $W_V$). Some works also include $W_K$ and $W_O$.

We first build a simple multi-head attention, then inject LoRA.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Standard multi-head attention."""
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.shape
        
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
    
    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class SmallTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int, max_len: int = 512):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        tok = self.tok_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        h = tok + pos
        
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        for layer in self.layers:
            h = layer(h, mask)
        
        h = self.norm(h)
        return self.head(h)

model = SmallTransformer(vocab_size=1000, d_model=256, n_heads=4, n_layers=4)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model params: {total_params:,}")

In [ ]:
def inject_lora(model: nn.Module, target_modules: list[str] = None,
                rank: int = 8, alpha: float = 16.0) -> nn.Module:
    """Inject LoRA into specified linear layers of a model.
    
    Args:
        model: the base model (will be modified in-place)
        target_modules: names of nn.Linear submodules to replace (e.g., ['W_q', 'W_v'])
        rank: LoRA rank
        alpha: LoRA scaling factor
    """
    if target_modules is None:
        target_modules = ['W_q', 'W_v']  # standard choice
    
    lora_layers = []
    for name, module in model.named_modules():
        for target in target_modules:
            if name.endswith(target):
                # Navigate to parent module
                parts = name.split('.')
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                
                # Replace with LoRA-wrapped version
                base_linear = getattr(parent, parts[-1])
                lora_layer = LoRALayer(base_linear, rank=rank, alpha=alpha)
                setattr(parent, parts[-1], lora_layer)
                lora_layers.append((name, lora_layer))
    
    # Freeze all non-LoRA parameters
    for param in model.parameters():
        param.requires_grad = False
    # Unfreeze LoRA parameters
    for name, lora_layer in lora_layers:
        lora_layer.lora_A.requires_grad = True
        lora_layer.lora_B.requires_grad = True
    
    return model, lora_layers


# Inject LoRA into W_q and W_v of all attention layers
lora_model = deepcopy(model)
lora_model, lora_layers = inject_lora(lora_model, target_modules=['W_q', 'W_v'], rank=8, alpha=16)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())

print(f"LoRA injected into {len(lora_layers)} layers:")
for name, layer in lora_layers:
    print(f"  {name}: rank={layer.rank}, params={layer.num_trainable_params:,}")

print(f"\nTrainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {trainable/total*100:.2f}%")

## 3. Merging LoRA Weights Back

At inference time, merge $\Delta W = (\alpha/r) \cdot BA$ into the base weight $W$. This gives **zero inference overhead** -- the model is the same architecture and size as the original.

In [ ]:
def merge_lora_weights(model: nn.Module) -> nn.Module:
    """Merge all LoRA layers back into base weights."""
    merged_model = deepcopy(model)
    
    for name, module in list(merged_model.named_modules()):
        if isinstance(module, LoRALayer):
            parts = name.split('.')
            parent = merged_model
            for p in parts[:-1]:
                parent = getattr(parent, p)
            
            merged_linear = module.merge_weights()
            setattr(parent, parts[-1], merged_linear)
    
    return merged_model


# Verify: merged model produces same output as LoRA model
test_input = torch.randint(0, 1000, (2, 32))

lora_model.eval()
with torch.no_grad():
    lora_out = lora_model(test_input)
    
    merged_model = merge_lora_weights(lora_model)
    merged_out = merged_model(test_input)

diff = (lora_out - merged_out).abs().max().item()
print(f"Max absolute difference after merging: {diff:.2e}")
print(f"Outputs match: {diff < 1e-5}")

# Verify merged model has no LoRA layers
has_lora = any(isinstance(m, LoRALayer) for m in merged_model.modules())
print(f"Merged model has LoRA layers: {has_lora}")

## 4. LoRA vs Full Fine-Tuning: Training Comparison

We train on a simple next-token prediction task to compare:
1. Full fine-tuning (all parameters)
2. LoRA fine-tuning (only LoRA parameters)

In [ ]:
# Create a tiny dataset: random sequences with a learnable pattern
def make_dataset(n_samples=500, seq_len=32, vocab_size=1000):
    """Simple pattern: token at position t depends on token at t-1."""
    data = torch.randint(0, vocab_size, (n_samples, seq_len))
    # Inject a pattern: every 4th token is (previous token + 1) % vocab_size
    for i in range(4, seq_len, 4):
        data[:, i] = (data[:, i-1] + 1) % vocab_size
    return data

dataset = make_dataset()
print(f"Dataset shape: {dataset.shape}")


def train_model(model, dataset, epochs=20, lr=1e-3, label=""):
    """Train with next-token prediction loss."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    model.train()
    losses = []
    
    for epoch in range(epochs):
        # Simple batching
        perm = torch.randperm(len(dataset))
        epoch_loss = 0
        n_batches = 0
        
        for i in range(0, len(dataset), 32):
            batch = dataset[perm[i:i+32]]
            inputs = batch[:, :-1]
            targets = batch[:, 1:]
            
            logits = model(inputs)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        if (epoch + 1) % 5 == 0:
            print(f"  [{label}] Epoch {epoch+1}: loss = {avg_loss:.4f}")
    
    return losses


# Full fine-tuning
print("=== Full Fine-Tuning ===")
full_model = SmallTransformer(vocab_size=1000, d_model=256, n_heads=4, n_layers=4)
full_trainable = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
print(f"Trainable params: {full_trainable:,}")
full_losses = train_model(full_model, dataset, epochs=20, lr=1e-3, label="Full")

# LoRA fine-tuning
print("\n=== LoRA Fine-Tuning (rank=8) ===")
lora_ft_model = SmallTransformer(vocab_size=1000, d_model=256, n_heads=4, n_layers=4)
lora_ft_model, _ = inject_lora(lora_ft_model, target_modules=['W_q', 'W_v'], rank=8, alpha=16)
lora_trainable = sum(p.numel() for p in lora_ft_model.parameters() if p.requires_grad)
print(f"Trainable params: {lora_trainable:,}")
lora_losses = train_model(lora_ft_model, dataset, epochs=20, lr=1e-3, label="LoRA")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
axes[0].plot(full_losses, 'b-o', label=f'Full FT ({full_trainable:,} params)', markersize=3)
axes[0].plot(lora_losses, 'r-s', label=f'LoRA r=8 ({lora_trainable:,} params)', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss: Full FT vs LoRA')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter comparison
methods = ['Full FT', 'LoRA r=4', 'LoRA r=8', 'LoRA r=16', 'LoRA r=64']
d = 256
param_counts = [
    full_trainable,
    8 * 2 * d * 4,    # 8 layers (4 blocks x W_q, W_v), rank 4
    8 * 2 * d * 8,    # rank 8
    8 * 2 * d * 16,   # rank 16
    8 * 2 * d * 64,   # rank 64
]
colors = ['steelblue'] + ['coral'] * 4
axes[1].bar(methods, param_counts, color=colors, edgecolor='white')
axes[1].set_ylabel('Parameter Count')
axes[1].set_title('Trainable Parameters by Method')
for i, v in enumerate(param_counts):
    axes[1].text(i, v + max(param_counts)*0.02, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Rank Selection Analysis

How does rank affect the expressiveness of the low-rank update? We examine the singular values of learned weight matrices to understand why low rank works.

In [ ]:
# Analyze singular value spectrum of a trained weight matrix
# This motivates why low-rank adaptation is sufficient

# Get a weight matrix from the trained full model
with torch.no_grad():
    W_pretrained = model.layers[0].attn.W_q.weight.clone()
    W_finetuned = full_model.layers[0].attn.W_q.weight.clone()
    delta_W = W_finetuned - W_pretrained

# SVD of the weight update
U, S, V = torch.svd(delta_W)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Singular values of delta_W
axes[0].plot(S.numpy(), 'b-')
axes[0].set_xlabel('Singular value index')
axes[0].set_ylabel('Singular value')
axes[0].set_title('Singular Values of Weight Update (delta_W)')
axes[0].axhline(y=S[8].item(), color='r', linestyle='--', label=f'Rank 8 cutoff')
axes[0].axhline(y=S[16].item(), color='g', linestyle='--', label=f'Rank 16 cutoff')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative energy
energy = (S ** 2).cumsum(0) / (S ** 2).sum()
axes[1].plot(energy.numpy(), 'b-')
axes[1].set_xlabel('Number of singular values')
axes[1].set_ylabel('Cumulative energy (fraction)')
axes[1].set_title('Cumulative Energy of Weight Update')
for r in [4, 8, 16, 32]:
    if r < len(energy):
        axes[1].axvline(x=r, color='r', linestyle=':', alpha=0.5)
        axes[1].text(r+1, energy[r].item()-0.05, f'r={r}: {energy[r].item():.1%}', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: weight updates during fine-tuning have low intrinsic rank.")
print("Most of the 'energy' is captured by the first few singular values.")

## 6. Quantization Concepts (QLoRA Intuition)

QLoRA = LoRA on a **quantized** base model. The base weights are stored in 4-bit (NF4), while LoRA adapters remain in float16/bfloat16.

### Quantization basics:
- **INT8**: map float values to 8-bit integers (256 levels)
- **INT4/NF4**: map to 4-bit (16 levels), NormalFloat4 uses quantile-based mapping
- **Double quantization**: quantize the quantization constants too

In [ ]:
def quantize_absmax_int8(tensor: torch.Tensor) -> tuple[torch.Tensor, float]:
    """Simple absmax INT8 quantization.
    Maps [-max, max] -> [-127, 127]
    """
    scale = tensor.abs().max() / 127.0
    quantized = torch.clamp(torch.round(tensor / scale), -127, 127).to(torch.int8)
    return quantized, scale

def dequantize_int8(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    return quantized.float() * scale


def quantize_absmax_int4(tensor: torch.Tensor) -> tuple[torch.Tensor, float]:
    """Simple absmax INT4 quantization.
    Maps [-max, max] -> [-7, 7]
    """
    scale = tensor.abs().max() / 7.0
    quantized = torch.clamp(torch.round(tensor / scale), -7, 7).to(torch.int8)  # stored in int8 container
    return quantized, scale

def dequantize_int4(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    return quantized.float() * scale


# Demonstrate quantization error
W = torch.randn(256, 256)  # simulate a weight matrix

# INT8
q8, s8 = quantize_absmax_int8(W)
W_int8 = dequantize_int8(q8, s8)
err_int8 = (W - W_int8).abs().mean().item()

# INT4
q4, s4 = quantize_absmax_int4(W)
W_int4 = dequantize_int4(q4, s4)
err_int4 = (W - W_int4).abs().mean().item()

print(f"Original: float32, {W.numel() * 4} bytes")
print(f"INT8: mean abs error = {err_int8:.6f}, {W.numel() * 1} bytes (4x compression)")
print(f"INT4: mean abs error = {err_int4:.6f}, ~{W.numel() // 2} bytes (8x compression)")

In [ ]:
# Visualize quantization error distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Original weight distribution
axes[0].hist(W.flatten().numpy(), bins=100, color='steelblue', alpha=0.7, density=True)
axes[0].set_title('Original Weights (FP32)')
axes[0].set_xlabel('Value')

# INT8 error
err8 = (W - W_int8).flatten().numpy()
axes[1].hist(err8, bins=100, color='coral', alpha=0.7, density=True)
axes[1].set_title(f'INT8 Quantization Error\n(mean |err| = {err_int8:.6f})')
axes[1].set_xlabel('Error')

# INT4 error
err4 = (W - W_int4).flatten().numpy()
axes[2].hist(err4, bins=100, color='orange', alpha=0.7, density=True)
axes[2].set_title(f'INT4 Quantization Error\n(mean |err| = {err_int4:.6f})')
axes[2].set_xlabel('Error')

plt.tight_layout()
plt.show()

## 7. Block-wise Quantization (used in QLoRA)

Per-tensor quantization wastes range on outliers. Block-wise quantization divides the tensor into blocks and quantizes each block independently.

In [ ]:
def quantize_blockwise_int4(tensor: torch.Tensor, block_size: int = 64):
    """Block-wise INT4 quantization (used in QLoRA via bitsandbytes).
    Each block of `block_size` values gets its own scale factor.
    """
    flat = tensor.flatten()
    n = flat.numel()
    # Pad to multiple of block_size
    if n % block_size != 0:
        pad_size = block_size - (n % block_size)
        flat = F.pad(flat, (0, pad_size))
    
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1).values / 7.0
    scales = scales.clamp(min=1e-8)  # avoid division by zero
    
    quantized = torch.clamp(
        torch.round(blocks / scales.unsqueeze(1)),
        -7, 7
    ).to(torch.int8)
    
    return quantized, scales, tensor.shape, n

def dequantize_blockwise_int4(quantized, scales, orig_shape, orig_numel):
    blocks = quantized.float() * scales.unsqueeze(1)
    flat = blocks.flatten()[:orig_numel]
    return flat.view(orig_shape)


# Compare per-tensor vs block-wise
W = torch.randn(256, 256)

# Per-tensor INT4
q4_pt, s4_pt = quantize_absmax_int4(W)
W_pt = dequantize_int4(q4_pt, s4_pt)
err_pt = (W - W_pt).abs().mean().item()

# Block-wise INT4
q4_bw, s4_bw, shape, n = quantize_blockwise_int4(W, block_size=64)
W_bw = dequantize_blockwise_int4(q4_bw, s4_bw, shape, n)
err_bw = (W - W_bw).abs().mean().item()

print(f"Per-tensor INT4 error: {err_pt:.6f}")
print(f"Block-wise INT4 error: {err_bw:.6f} (block_size=64)")
print(f"Block-wise reduction: {(1 - err_bw/err_pt)*100:.1f}% less error")
print(f"\nOverhead: {len(s4_bw)} scale factors for {W.numel()} weights")
print(f"Scale overhead: {len(s4_bw) * 2 / (W.numel() // 2) * 100:.2f}% (FP16 scales)")

## 8. QLoRA: Putting It Together

QLoRA workflow:
1. Quantize base model to 4-bit (NF4)
2. Add LoRA adapters in float16/bfloat16
3. During forward: dequantize base weights on-the-fly, add LoRA output
4. Only LoRA gradients are computed (base weights are frozen and quantized)

In [ ]:
class QLoRALayer(nn.Module):
    """Simplified QLoRA layer.
    Base weight is quantized to INT4, LoRA adapters remain in float.
    """
    def __init__(self, base_layer: nn.Linear, rank: int = 8, alpha: float = 16.0,
                 block_size: int = 64):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.block_size = block_size
        
        d_out, d_in = base_layer.weight.shape
        self.d_out = d_out
        self.d_in = d_in
        
        # Quantize base weight to INT4 (block-wise)
        q_weight, scales, shape, n = quantize_blockwise_int4(base_layer.weight.data, block_size)
        self.register_buffer('q_weight', q_weight)
        self.register_buffer('scales', scales)
        self.register_buffer('weight_shape', torch.tensor(list(shape)))
        self.register_buffer('weight_numel', torch.tensor(n))
        
        # Bias (if any)
        if base_layer.bias is not None:
            self.register_buffer('bias', base_layer.bias.data.clone())
        else:
            self.bias = None
        
        # LoRA adapters (full precision)
        self.lora_A = nn.Parameter(torch.empty(rank, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dequantize base weight on-the-fly
        W = dequantize_blockwise_int4(
            self.q_weight, self.scales,
            tuple(self.weight_shape.tolist()),
            self.weight_numel.item()
        )
        
        # Base forward
        base_out = F.linear(x, W, self.bias)
        
        # LoRA forward
        lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B)
        
        return base_out + self.scaling * lora_out


# Demo: memory comparison
base_linear = nn.Linear(1024, 1024)
qlora_layer = QLoRALayer(base_linear, rank=16, alpha=32)

# Memory estimate
fp32_bytes = base_linear.weight.numel() * 4  # float32
int4_bytes = qlora_layer.q_weight.numel() // 2  # 4 bits each (stored in int8 container but conceptually 4-bit)
scale_bytes = qlora_layer.scales.numel() * 2  # float16 scales
lora_bytes = (qlora_layer.lora_A.numel() + qlora_layer.lora_B.numel()) * 4  # float32

print(f"Full FP32: {fp32_bytes:,} bytes")
print(f"QLoRA base (INT4): ~{int4_bytes:,} bytes + {scale_bytes:,} scale bytes")
print(f"QLoRA adapters: {lora_bytes:,} bytes")
print(f"QLoRA total: ~{int4_bytes + scale_bytes + lora_bytes:,} bytes")
print(f"Compression: {fp32_bytes / (int4_bytes + scale_bytes + lora_bytes):.1f}x")

## 9. Rank vs Performance Tradeoff

In [ ]:
# Train LoRA with different ranks and compare
ranks = [1, 2, 4, 8, 16, 32]
final_losses = []
param_counts = []

for r in ranks:
    m = SmallTransformer(vocab_size=1000, d_model=256, n_heads=4, n_layers=4)
    m, _ = inject_lora(m, target_modules=['W_q', 'W_v'], rank=r, alpha=2*r)
    
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    param_counts.append(trainable)
    
    losses = train_model(m, dataset, epochs=15, lr=1e-3, label=f"r={r}")
    final_losses.append(losses[-1])
    print(f"  rank={r}, params={trainable:,}, final_loss={losses[-1]:.4f}\n")

fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = 'steelblue'
ax1.set_xlabel('LoRA Rank')
ax1.set_ylabel('Final Loss', color=color1)
ax1.plot(ranks, final_losses, 'o-', color=color1, linewidth=2, markersize=8)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'coral'
ax2.set_ylabel('Trainable Params', color=color2)
ax2.bar(ranks, param_counts, alpha=0.3, color=color2, width=[r*0.3 for r in ranks])
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('LoRA Rank vs Performance vs Parameter Count')
plt.tight_layout()
plt.show()

## 10. Which Layers to Apply LoRA To?

| Configuration | Typical Use | Notes |
|--------------|-------------|-------|
| W_q, W_v only | Default (original LoRA paper) | Good baseline |
| W_q, W_k, W_v, W_o | Better performance | 2x LoRA params vs default |
| All linear layers | Best performance (used in QLoRA paper) | Includes FFN layers |
| Embedding + LM head | Sometimes added for vocab adaptation | Domain-specific fine-tuning |

In [ ]:
# Compare different LoRA targets
configs = {
    'W_q, W_v (default)': ['W_q', 'W_v'],
    'All attention': ['W_q', 'W_k', 'W_v', 'W_o'],
}

results = {}
for name, targets in configs.items():
    m = SmallTransformer(vocab_size=1000, d_model=256, n_heads=4, n_layers=4)
    m, layers = inject_lora(m, target_modules=targets, rank=8, alpha=16)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    
    print(f"\n{name}: {trainable:,} trainable params, {len(layers)} LoRA layers")
    losses = train_model(m, dataset, epochs=15, lr=1e-3, label=name)
    results[name] = {'losses': losses, 'params': trainable}

plt.figure(figsize=(10, 5))
for name, data in results.items():
    plt.plot(data['losses'], '-o', label=f"{name} ({data['params']:,})", markersize=4)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LoRA Target Layer Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Memory Footprint Comparison at Scale

Real-world memory estimates for a 7B parameter model:

In [ ]:
# Memory estimates for a 7B model
d_model = 4096
n_layers = 32
n_params_7b = 7e9

# Full fine-tuning: model (fp16) + optimizer (Adam: 2x fp32 states) + gradients (fp16)
full_ft_gb = (n_params_7b * 2 + n_params_7b * 4 * 2 + n_params_7b * 2) / 1e9

# LoRA: model (fp16) + LoRA params + optimizer for LoRA only
lora_params_per_layer = 2 * d_model * 16 * 2  # W_q, W_v, rank=16, A+B
total_lora_params = lora_params_per_layer * n_layers
lora_gb = (n_params_7b * 2 + total_lora_params * 2 + total_lora_params * 4 * 2 + total_lora_params * 2) / 1e9

# QLoRA: model (4-bit) + LoRA params + optimizer for LoRA only
qlora_gb = (n_params_7b * 0.5 + total_lora_params * 2 + total_lora_params * 4 * 2 + total_lora_params * 2) / 1e9

methods = ['Full FT\n(FP16+Adam)', 'LoRA\n(FP16 base)', 'QLoRA\n(4-bit base)']
memory = [full_ft_gb, lora_gb, qlora_gb]
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, memory, color=colors, edgecolor='white', linewidth=2)
ax.set_ylabel('GPU Memory (GB)')
ax.set_title('Memory Requirements for 7B Model Fine-tuning')

for bar, mem in zip(bars, memory):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{mem:.1f} GB', ha='center', fontweight='bold', fontsize=12)

# Add GPU reference lines
ax.axhline(y=24, color='gray', linestyle='--', alpha=0.5)
ax.text(2.4, 24.5, 'RTX 3090 (24GB)', fontsize=9, color='gray')
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5)
ax.text(2.4, 80.5, 'A100 (80GB)', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print(f"LoRA trainable params: {total_lora_params:,} ({total_lora_params/n_params_7b*100:.3f}% of 7B)")
print(f"QLoRA enables 7B fine-tuning on a single consumer GPU!")

## Interview Notes

**"Why does low-rank adaptation work?"**
- The weight updates during fine-tuning have low intrinsic dimensionality
- Aghajanyan et al. (2021) showed that pre-trained models have a low "intrinsic dimension" -- you only need to update a small subspace
- The SVD of actual weight updates shows rapid singular value decay
- Intuitively: fine-tuning adapts to a specific task, which requires much less capacity than pre-training

**"How many parameters does LoRA add?"**
- Per layer: $2 \times d \times r$ (matrix A: $r \times d$, matrix B: $d \times r$)
- For all Q,V projections in a 7B model with rank 16: ~0.1% of total params
- Example: LLaMA-7B (d=4096, 32 layers, Q+V): $32 \times 2 \times 2 \times 4096 \times 16 = 8.4M$ params

**"How to choose rank?"**
- Typical range: 4-64 (8-16 is common default)
- Higher rank = more expressiveness but more params and memory
- Task complexity matters: simple tasks (sentiment) need rank 4-8, complex tasks (code gen) may need 32-64
- Alpha is typically set to 2x rank (alpha = 2r)

**"Which layers to apply LoRA to?"**
- Original paper: W_q, W_v (attention only)
- QLoRA paper found: all linear layers works best
- Practical default: W_q, W_k, W_v, W_o (all attention projections)
- Adding FFN layers helps for harder tasks

**"What is QLoRA?"**
- Base model quantized to 4-bit NormalFloat (NF4)
- LoRA adapters in bfloat16
- Double quantization: quantize the quantization constants
- Paged optimizers: handle memory spikes via CPU offload
- Result: 7B model fine-tuning on a single 24GB GPU

**Typical ranks used in practice:**
- LLaMA fine-tuning: r=8 to r=64
- Stable Diffusion LoRA: r=4 to r=128
- Production recommendation: start with r=16, tune if needed